# Dataset comparison

Statistiken über das reale BraTS-2023 Trainset vs. generierte Samples. Reale Stats werden gecacht (`output/real_dataset_stats.json`).

Die Notebooks für die Darstellung wurden größtenteils mit KI erstellt

## Pfade

In [ ]:
from pathlib import Path

BRATS_REAL_ROOT = Path("/home/sven/Desktop/diffsuion/Processed/brats/32")
BRATS_SEG_ROOT  = Path("/home/sven/Desktop/diffsuion/Processed/brats/32_seg")
SAMPLES_ROOT    = Path("/home/sven/Desktop/diffsuion/Sample Generation")
OUTPUT_DIR      = Path.cwd() / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

FID_BEST_SAMPLER = {
    "baseline":          "DDPM",
    "data_augmentation": "UniPC",
    "linear_schedule":   "DDPM",
    "no_attention":      "DDPM",
    "noise_prediction":  "DDPM",
}
PIPELINE_SAMPLER = "UniPC"

print(OUTPUT_DIR)

## Style

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import nibabel as nib
from tqdm.auto import tqdm

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Latin Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "lines.linewidth": 1.2,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

MODEL_COLORS = {
    "baseline":          "#1f77b4",
    "data_augmentation": "#ff7f0e",
    "linear_schedule":   "#d62728",
    "no_attention":      "#2ca02c",
    "noise_prediction":  "#9467bd",
    "baseline_48":       "#17becf",
    "pipeline":          "#8c564b",
}

MODEL_LABELS = {
    "baseline":          r"Baseline $32^3$",
    "data_augmentation": "Data Augmentation",
    "linear_schedule":   "Linearer Schedule",
    "no_attention":      "Ohne Self-Attention",
    "noise_prediction":  "Noise-Prediction",
    "baseline_48":       r"Baseline $48^3$",
    "pipeline":          "Pipeline (T1N)",
}


def load_volume(p):
    return nib.load(str(p)).get_fdata().astype(np.float32)

## Reale Statistik (gecacht)

In [ ]:
REAL_STATS_CACHE = OUTPUT_DIR / "real_dataset_stats.json"
BIN_EDGES = np.linspace(-1.0, 1.0, 51)


def compute_real_stats():
    vols = sorted(BRATS_REAL_ROOT.glob("*.nii.gz"))
    segs = sorted(BRATS_SEG_ROOT.glob("*.nii.gz"))
    print(f"{len(vols)} T1N, {len(segs)} Masken")

    hist = np.zeros(len(BIN_EDGES) - 1, dtype=np.int64)
    vsum = vsq = 0.0
    vn = 0
    for vp in tqdm(vols, desc="T1N"):
        v = load_volume(vp).ravel()
        h, _ = np.histogram(v, bins=BIN_EDGES)
        hist += h
        vsum += float(v.sum())
        vsq  += float((v.astype(np.float64) ** 2).sum())
        vn   += v.size

    mean = vsum / vn
    std  = float(np.sqrt(vsq / vn - mean ** 2))

    total_voxels = []
    class_voxels = {1: [], 2: [], 3: []}
    bbox = []
    n_empty = 0
    for sp in tqdm(segs, desc="Seg"):
        seg = load_volume(sp).astype(np.int8)
        total = int((seg > 0).sum())
        total_voxels.append(total)
        if total == 0:
            n_empty += 1
            continue
        for lab in (1, 2, 3):
            class_voxels[lab].append(int((seg == lab).sum()))
        nz = np.where(seg > 0)
        bbox.append([int(nz[0].max() - nz[0].min() + 1),
                     int(nz[1].max() - nz[1].min() + 1),
                     int(nz[2].max() - nz[2].min() + 1)])

    return {
        "n_volumes": len(vols),
        "n_segmentations": len(segs),
        "n_empty_segmentations": n_empty,
        "intensity": {
            "bin_edges": BIN_EDGES.tolist(),
            "counts": hist.tolist(),
            "global_mean": mean,
            "global_std": std,
        },
        "tumor": {
            "total_voxels": total_voxels,
            "class_voxels": {str(k): v for k, v in class_voxels.items()},
            "bbox_dims": bbox,
            "total_mean": float(np.mean(total_voxels)),
            "total_median": float(np.median(total_voxels)),
            "total_min": int(np.min(total_voxels)),
            "total_max": int(np.max(total_voxels)),
            "class_mean": {str(k): float(np.mean(v)) for k, v in class_voxels.items() if v},
        },
    }


if REAL_STATS_CACHE.exists():
    real_stats = json.load(REAL_STATS_CACHE.open())
    print(f"cache: {REAL_STATS_CACHE.name}")
else:
    real_stats = compute_real_stats()
    with REAL_STATS_CACHE.open("w") as f:
        json.dump(real_stats, f, indent=2)

print(f"N={real_stats['n_volumes']}, "
      f"mean={real_stats['intensity']['global_mean']:.4f}, "
      f"std={real_stats['intensity']['global_std']:.4f}")
print(f"Tumor mean={real_stats['tumor']['total_mean']:.1f}, "
      f"median={real_stats['tumor']['total_median']:.1f}")

## Intensitätsverteilung

In [ ]:
def load_intensity_hist(sampler, model=None, size=32, pipeline=False):
    if pipeline:
        p = SAMPLES_ROOT / sampler / str(size) / "samples_pipeline" / "stats.json"
        key = "brain_voxel_distribution"
    else:
        p = SAMPLES_ROOT / sampler / str(size) / f"samples_{model}" / "stats.json"
        key = "voxel_distribution"
    if not p.exists():
        return None
    vd = json.load(p.open()).get("aggregates", {}).get(key)
    if vd is None:
        return None
    return np.array(vd["bin_edges"]), np.array(vd["counts"])


def to_density(edges, counts):
    w = np.diff(edges)
    s = counts.sum()
    return (counts / s) / w if s > 0 else counts


edges_r = np.array(real_stats["intensity"]["bin_edges"])
counts_r = np.array(real_stats["intensity"]["counts"])
centers_r = 0.5 * (edges_r[:-1] + edges_r[1:])
density_r = to_density(edges_r, counts_r)

samplers = ["DDPM", "DDIM", "DPMSolverPP", "UniPC"]
SAMPLER_LABELS = {"DDPM": "DDPM (250)", "DDIM": "DDIM (50)",
                  "DPMSolverPP": "DPM-Solver++ (25)", "UniPC": "UniPC (10)"}

models_32 = ["baseline", "data_augmentation", "linear_schedule", "no_attention", "noise_prediction"]

fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=True, sharey=True)
flat = axes.flatten()

for ax, sampler in zip(flat, samplers):
    ax.fill_between(centers_r, density_r, color="gray", alpha=0.35, label="BraTS (real)")
    ax.plot(centers_r, density_r, color="black", linewidth=1.0)
    for m in models_32:
        res = load_intensity_hist(sampler, m, size=32)
        if res is None:
            continue
        edges, counts = res
        centers = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centers, to_density(edges, counts),
                color=MODEL_COLORS[m], linewidth=1.1, label=MODEL_LABELS[m])
    res48 = load_intensity_hist(sampler, "baseline", size=48)
    if res48 is not None:
        edges, counts = res48
        centers = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centers, to_density(edges, counts),
                color=MODEL_COLORS["baseline_48"], linewidth=1.1,
                label=MODEL_LABELS["baseline_48"])
    res_pipe = load_intensity_hist(sampler, pipeline=True)
    if res_pipe is not None:
        edges, counts = res_pipe
        centers = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centers, to_density(edges, counts),
                color=MODEL_COLORS["pipeline"], linewidth=1.4, linestyle="--",
                label=MODEL_LABELS["pipeline"])
    ax.set_title(SAMPLER_LABELS[sampler], fontsize=10)
    ax.set_xlim(-1.0, 1.0)
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.3, linewidth=0.5)

for ax in axes[-1, :]:
    ax.set_xlabel("Voxel-Intensität")
for ax in axes[:, 0]:
    ax.set_ylabel("Dichte (log)")

# Legenden-Reihenfolge explizit setzen (matcht Tabellen-Reihenfolge im Kapitel).
legend_order = [
    "BraTS (real)",
    MODEL_LABELS["baseline"],
    MODEL_LABELS["data_augmentation"],
    MODEL_LABELS["linear_schedule"],
    MODEL_LABELS["no_attention"],
    MODEL_LABELS["noise_prediction"],
    MODEL_LABELS["baseline_48"],
    MODEL_LABELS["pipeline"],
]
handles, labels = flat[0].get_legend_handles_labels()
lookup = dict(zip(labels, handles))
ordered_handles = [lookup[l] for l in legend_order if l in lookup]
ordered_labels  = [l for l in legend_order if l in lookup]
fig.legend(ordered_handles, ordered_labels, loc="lower center", ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, -0.01), frameon=True)

fig.suptitle("Voxel-Intensitätsverteilung pro Sampler — Modelle und Pipeline vs. BraTS",
             fontsize=11, y=0.985)
fig.tight_layout(rect=[0, 0.06, 1, 0.95])
fig.savefig(OUTPUT_DIR / "intensity_histogram_real_vs_generated.pdf")
plt.show()

## Intensitätsverteilung — Einzelpanels

In [ ]:
# Detail-Grid: Zeilen = Sampler, Spalten = Modelle (+ Baseline 48 + Pipeline).
# Pro Panel BraTS als grauer Hintergrund und das einzelne Modell darüber.
detail_models = [
    ("baseline",          32),
    ("data_augmentation", 32),
    ("linear_schedule",   32),
    ("no_attention",      32),
    ("noise_prediction",  32),
    ("baseline_48",       48),
    ("pipeline",          32),
]

n_rows = len(samplers)
n_cols = len(detail_models)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.9, n_rows * 2.0),
                         sharex=True, sharey=True)

for r, sampler in enumerate(samplers):
    for c, (m, size) in enumerate(detail_models):
        ax = axes[r, c]
        ax.fill_between(centers_r, density_r, color="gray", alpha=0.35)
        ax.plot(centers_r, density_r, color="black", linewidth=0.8)
        if m == "pipeline":
            res = load_intensity_hist(sampler, pipeline=True)
        elif m == "baseline_48":
            res = load_intensity_hist(sampler, "baseline", size=48)
        else:
            res = load_intensity_hist(sampler, m, size=size)
        if res is not None:
            edges, counts = res
            centers = 0.5 * (edges[:-1] + edges[1:])
            ax.plot(centers, to_density(edges, counts),
                    color=MODEL_COLORS[m], linewidth=1.3)
        ax.set_xlim(-1.0, 1.0)
        ax.set_yscale("log")
        ax.grid(True, which="both", alpha=0.3, linewidth=0.4)
        if r == 0:
            ax.set_title(MODEL_LABELS[m], fontsize=8)
    axes[r, 0].set_ylabel(SAMPLER_LABELS[sampler], fontsize=9)

for ax in axes[-1, :]:
    ax.set_xlabel("Voxel-Intensität")

fig.suptitle("Voxel-Intensitätsverteilung pro Sampler × Modell vs. BraTS",
             fontsize=11, y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(OUTPUT_DIR / "intensity_histogram_detail.pdf")
plt.show()

## Tumor-Größe

In [ ]:
def pipeline_tumor_sizes(sampler, size=32):
    p = SAMPLES_ROOT / sampler / str(size) / "samples_pipeline" / "stats.json"
    if not p.exists():
        return None
    ps = json.load(p.open()).get("per_sample", [])
    total = np.array([s.get("tumor_voxel_count", 0) for s in ps])
    classes = {1: [], 2: [], 3: []}
    for s in ps:
        for i, v in enumerate(s.get("tumor_voxel_count_per_class", [0, 0, 0]), start=1):
            classes[i].append(v)
    return total, {k: np.array(v) for k, v in classes.items()}


real_total = np.array(real_stats["tumor"]["total_voxels"])

pipe_data = {s: pipeline_tumor_sizes(s, 32) for s in samplers}
gen_max = max((d[0].max() for d in pipe_data.values() if d is not None), default=0)
bins = np.linspace(0, max(real_total.max(), gen_max), 40)

fig, axes = plt.subplots(2, 2, figsize=(8.5, 7), sharex=True, sharey=True)
flat = axes.flatten()

for ax, sampler in zip(flat, samplers):
    pipe = pipe_data[sampler]
    if pipe is None:
        ax.text(0.5, 0.5, "keine Daten", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(sampler, fontsize=10)
        continue
    gen_total, _ = pipe
    ax.hist(real_total, bins=bins, alpha=0.5, color="gray",
            label=f"BraTS, N={len(real_total)}", density=True)
    ax.hist(gen_total, bins=bins, alpha=0.55, color=MODEL_COLORS["baseline"],
            label=f"Pipeline, N={len(gen_total)}", density=True)
    ax.set_title(SAMPLER_LABELS[sampler], fontsize=10)
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.legend(loc="upper right", fontsize=7)

for ax in axes[-1, :]:
    ax.set_xlabel("Tumor-Voxelanzahl (gesamt)")
for ax in axes[:, 0]:
    ax.set_ylabel("Dichte")

fig.suptitle("Pipeline-Tumor-Voxelanzahl vs. BraTS pro Sampler",
             fontsize=11, y=0.985)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(OUTPUT_DIR / "tumor_size_distribution.pdf")
plt.show()

for sampler in samplers:
    if pipe_data[sampler] is None:
        continue
    g = pipe_data[sampler][0]
    print(f"{sampler:>11s}: mean={g.mean():7.1f}  median={np.median(g):7.1f}  max={g.max()}")
print(f"{'BraTS':>11s}: mean={real_total.mean():7.1f}  median={np.median(real_total):7.1f}  max={real_total.max()}")

## LaTeX-Tabelle: Tumor-Subregionen pro Sampler

In [ ]:
def fmt_de(value, decimals=1):
    return f"{value:.{decimals}f}".replace(".", "{,}")


def fmt_pair(mean, std, decimals=1):
    return f"{fmt_de(mean, decimals)} \\pm {fmt_de(std, decimals)}"


real_classes = {int(k): np.array(v) for k, v in real_stats["tumor"]["class_voxels"].items()}

rows = []
real_row = ["BraTS (real)"]
for cls in (1, 2, 3):
    arr = real_classes[cls]
    real_row.append(fmt_pair(arr.mean(), arr.std()))
rows.append(real_row)

for sampler in samplers:
    pipe = pipeline_tumor_sizes(sampler, 32)
    if pipe is None:
        continue
    _, gen_classes = pipe
    row = [sampler]
    for cls in (1, 2, 3):
        arr = gen_classes[cls]
        row.append(fmt_pair(arr.mean(), arr.std()))
    rows.append(row)

lines = [
    r"\begin{tabular}{lccc}",
    r"\toprule",
    r"Konfiguration & NCR (Label 1) & ED (Label 2) & ET (Label 3) \\",
    r"\midrule",
]
for row in rows:
    lines.append(f"{row[0]} & ${row[1]}$ & ${row[2]}$ & ${row[3]}$ \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]

latex_subregions = "\n".join(lines)
print(latex_subregions)